In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
df = pd.read_csv('../data/bank/bank-full.csv', sep=';')

In [10]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [11]:
df.shape

(45211, 17)

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

class PdaysImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.median_ = None
        
    def fit(self, X, y=None):
        pdays_series = X['pdays']
        self.median_ = pdays_series[pdays_series != -1].median()
        return self
    
    def transform(self, X):
        X_transformed = X.copy()

        X_transformed['pdays_not_contacted'] = (X_transformed['pdays'] == -1).astype(int)

        X_transformed['pdays'] = X_transformed['pdays'].replace(-1, self.median_)

        return X_transformed[['pdays', 'pdays_not_contacted']]

    def get_feature_names_out(self, input_features=None):
        return np.array(['pdays', 'pdays_not_contacted'])

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_cols = X_train.select_dtypes(include=['object', 'str']).columns.tolist()
pdays_col = ['pdays']

numeric_cols = [col for col in X_train.columns if col not in categorical_cols + pdays_col]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), categorical_cols),
        ('pdays_custom', PdaysImputer(), pdays_col),
        ('num', 'passthrough', numeric_cols) 
    ],
    remainder='drop'
)

preprocessor.set_output(transform="pandas")

X_train_final = preprocessor.fit_transform(X_train)

X_test_final = preprocessor.transform(X_test)

In [15]:
X_train_final.columns

Index(['cat__job_blue-collar', 'cat__job_entrepreneur', 'cat__job_housemaid',
       'cat__job_management', 'cat__job_retired', 'cat__job_self-employed',
       'cat__job_services', 'cat__job_student', 'cat__job_technician',
       'cat__job_unemployed', 'cat__job_unknown', 'cat__marital_married',
       'cat__marital_single', 'cat__education_secondary',
       'cat__education_tertiary', 'cat__education_unknown', 'cat__default_yes',
       'cat__housing_yes', 'cat__loan_yes', 'cat__contact_telephone',
       'cat__contact_unknown', 'cat__month_aug', 'cat__month_dec',
       'cat__month_feb', 'cat__month_jan', 'cat__month_jul', 'cat__month_jun',
       'cat__month_mar', 'cat__month_may', 'cat__month_nov', 'cat__month_oct',
       'cat__month_sep', 'cat__poutcome_other', 'cat__poutcome_success',
       'cat__poutcome_unknown', 'pdays_custom__pdays',
       'pdays_custom__pdays_not_contacted', 'num__age', 'num__balance',
       'num__day', 'num__duration', 'num__campaign', 'num__previous'

In [16]:
X_train_final.shape

(36168, 43)

In [17]:
X_test_final.shape

(9043, 43)

In [18]:

pdays_col = 'pdays_custom__pdays'
indicator_col = 'pdays_custom__pdays_not_contacted'

print("=== TRAINING DATA SANITY CHECK ===")

imputed_train_values = X_train_final.loc[X_train_final[indicator_col] == 1, pdays_col].unique()
print(f"Unique 'pdays' values where indicator is 1: {imputed_train_values}")

print(f"NaNs in 'pdays': {X_train_final[pdays_col].isna().sum()}")
print(f"Total NaNs in entire X_train_final: {X_train_final.isna().sum().sum()}")


print("\n=== TEST DATA SANITY CHECK ===")

imputed_test_values = X_test_final.loc[X_test_final[indicator_col] == 1, pdays_col].unique()
print(f"Unique 'pdays' values where indicator is 1: {imputed_test_values}")

print(f"NaNs in 'pdays': {X_test_final[pdays_col].isna().sum()}")
print(f"Total NaNs in entire X_test_final: {X_test_final.isna().sum().sum()}")

=== TRAINING DATA SANITY CHECK ===
Unique 'pdays' values where indicator is 1: [194]
NaNs in 'pdays': 0
Total NaNs in entire X_train_final: 0

=== TEST DATA SANITY CHECK ===
Unique 'pdays' values where indicator is 1: [194]
NaNs in 'pdays': 0
Total NaNs in entire X_test_final: 0


RobustScaler uses the median and IQR, which are less sensitive to extreme observations than the mean and standard deviation used by StandardScaler.

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'pdays' in numeric_cols:
    numeric_cols.remove('pdays')

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), categorical_cols),
        ('pdays_custom', PdaysImputer(), pdays_col),
        ('num', RobustScaler(), numeric_cols) 
    ],
    remainder='drop'
)
preprocessor_scaled.set_output(transform="pandas")

preprocessor_unscaled = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), categorical_cols),
        ('pdays_custom', PdaysImputer(), pdays_col),
        ('num', 'passthrough', numeric_cols)
    ],
    remainder='drop'
)
preprocessor_unscaled.set_output(transform="pandas")

print(f"Numerical columns to be processed: {numeric_cols}")
print("Preprocessors are ready!")

Numerical columns to be processed: ['age', 'balance', 'day', 'duration', 'campaign', 'previous']
Preprocessors are ready!


In [22]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [23]:
numeric_cols

['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

In [24]:
if 'duration' in numeric_cols:
    numeric_cols.remove('duration')

if 'pdays' in numeric_cols:
    numeric_cols.remove('pdays')

In [25]:
print(numeric_cols)

['age', 'balance', 'day', 'campaign', 'previous']
